In [2]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import torch.nn as nn

In [3]:
#defining tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

print(f"Tokenizer vocab size:", {tokenizer.vocab_size})
print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer vocab size: {30522}
Model loaded


In [4]:
#testing the tokenizer
sample_text = "Buy now and save 50% today"

tokens = tokenizer(
    sample_text,
    return_tensors = 'pt', #return tensors
    max_length = 128,
    padding = 'max_length', #short sequence -> 128
    truncation = True #cut sequence longer than 128
)

print("input_ids shape: ", tokens['input_ids'].shape)
print('attention_mask shape: ', tokens['attention_mask'].shape)
print()
print('input_ids: ', tokens['input_ids'])
print('attention_mask: ', tokens['attention_mask'])
print()

print('Decoded: ', tokenizer.decode(tokens['input_ids'][0])) #decoding

input_ids shape:  torch.Size([1, 128])
attention_mask shape:  torch.Size([1, 128])

input_ids:  tensor([[ 101, 4965, 2085, 1998, 3828, 2753, 1003, 2651,  102,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])
attention_mask:  tensor([[1, 1, 1, 

In [7]:
#text embedder
class TextEmbedder(nn.Module):
  def __init__(self):
    super().__init__()

    self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased') #loading distilbert

    for param in self.bert.parameters():
      param.requires_grad = False

    self.embedding_head = nn.Sequential(
        nn.Linear(768, 768),
        nn.ReLU(),
        nn.Dropout(0.1)
    )

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(
        input_ids = input_ids,
        attention_mask = attention_mask
    )

    cls_token = outputs.last_hidden_state[:,0,:] #extract CLS token (position 0 of last hidden state)

    embedding = self.embedding_head(cls_token)

    return embedding

text_embedder = TextEmbedder()
print(f"Embedder Readey")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder Readey


In [9]:
#testing with real data
ad_texts = ["Summer Sale - 50% off on all products. Shop now before its too late.",
            "Increase your ROI from your ads by using AdLens. Try it today.",
            "The best travel deals of 2025. Book flights and hotels in one click."]

tokens = tokenizer(
    ad_texts,
    return_tensors = 'pt',
    max_length = 128,
    padding = 'max_length',
    truncation = True
)

print("Batch Input Ids Shape:", tokens['input_ids'].shape)

text_embedder.eval() #capturing the embeddings
with torch.no_grad(): #disabling dropouts
  embeddings = text_embedder(
      tokens['input_ids'],
      tokens['attention_mask']
  )

print(f"Embeddings Shape:  {embeddings.shape}")
print(f"All Finite:  {torch.isfinite(embeddings).all()}")


Batch Input Ids Shape: torch.Size([3, 128])
Embeddings Shape:  torch.Size([3, 768])
All Finite:  True
